In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Compact12 四片定向恢复

选择 GPU 后直接 Run all。先恢复已保存原始观测及 CPU 后评，再补限定视图/配对素材，最后按原分母汇总四片。原缺口为 87 个逻辑视图和 11 个质量阶段；经授权 ucf-0643 PixelSeal 正例完整替代，另测原 9 个完整视图及 3 个质量阶段，因此本次执行范围为 96 视图、14 质量阶段、79 个共享原始视图组，来自 6 个源。原始视图组数不等于 GPU 调用数，全局分母不增加。原四片已停止，旧成功、缺失、失败、预算和 v4 结果全部保留。


In [ ]:
SOURCE_REF = 'main'
SOURCE_DIRECTORY = None
PREVIOUS_RUNTIME_STOPPED = True  # 已确认原四片 runtime 均 CLOSED
CONFIG = dict(
    parent='/content/drive/MyDrive/Video-WM_Result/baselines/public-native-frameindex-compact12-v1/attempt01',
    output='/content/drive/MyDrive/Video-WM_Result/baselines/public-native-frameindex-compact12-v1/attempt01/recovery/repair01',
    cache='/content/drive/MyDrive/Video-WM_Cache/baseline-compact12-v1',
    old='/content/drive/MyDrive/Video-WM_Result/baselines/public-native-and-frameindex-fpr001-v4/attempt01',
    old_cache='/content/drive/MyDrive/Video-WM_Cache/baseline-v4')


自动导入同目录的 `local_cpu_recovery.zip`，仅补缺失文件并将本地 CPU 成本计入一次；写入前检查原分片和修复目录的运行状态，采用完整临时写入后原子发布，已有 Colab 记录不覆盖，不需要手工上传。已恢复的 3 视图和 5 质量阶段直接跳过。PixelSeal 与 FrameIndex 的 OFF 原始读取共享，分别执行原逻辑后评。缺 epoch 时仅在回执能唯一关联时恢复，绝不混 epoch 或按分数择优。ucf-0642 的两个正例臂各 12 视图全缺，按原查询整臂重建；ucf-0643 PixelSeal 按原方法/消息完整重建 12 视图和配套 3 质量，新臂每槽均只认新结果，失败槽保持未知，绝不回填旧 9 个成功视图。旧臂仍保留，替代关系记录在 `shards/part2/replacements/`。其余部分 epoch 输入丢失、已提交数组不可读、关联不清等单列未决。质量必要 codec 成本独立记录；LPIPS 不补算。


In [ ]:
from pathlib import Path
import urllib.request, importlib.util
name = 'native_baseline_compact_recovery_v1_entry.py'
if SOURCE_DIRECTORY:
    path = Path(SOURCE_DIRECTORY)/'scripts'/name
else:
    path = Path('/content')/name
    urllib.request.urlretrieve(f'https://raw.githubusercontent.com/RICHAAARC/SC-SSTW/{SOURCE_REF}/scripts/{name}', path)
spec = importlib.util.spec_from_file_location('compact_repair_entry', path)
entry = importlib.util.module_from_spec(spec)
spec.loader.exec_module(entry)
RUN = entry.execute(CONFIG, SOURCE_DIRECTORY, SOURCE_REF, PREVIOUS_RUNTIME_STOPPED)


新恢复包位于正式结果的 `recovery/repair01`，旧分片仍只读。`cpu_recovery.json` 保存逐槽恢复/剩余清单，`physical_recovery.json` 保存补算返回与失败，`global/tables` 保存全局 A/B/C/D 表。运行完成、测量完整和科学结论分别报告；保留原阈值、CAL32/TEST500、12 视图、DCT2＋CDC32 和旧 H/L。


In [ ]:
print('返回:', RUN)
print('全局表目录:', CONFIG['output']+'/global/tables')
